In [ ]:
import json
from pathlib import Path

import nibabel as nib
import numpy as np
from scipy import ndimage
from skimage.segmentation import clear_border
from skimage import measure
from skimage.measure import label
from skimage.feature import blob_log
from tqdm.notebook import tqdm
import pandas as pd
from skimage.filters import sato

MIN_LUNG_HU = -950
MAX_LUNG_HU = -650
MIN_NODULE_HU = -150
MAX_NODULE_HU = 200

In [2]:
json_path = Path("/home/chest_ct/code/data/rexgrounding-ct/dataset_2d_filtered.json")

volume_root = Path("/home/chest_ct/code/data/data_volumes/dataset/train_fixed")
mask_root = Path("/home/chest_ct/code/data/segmentations/segmentations")

output_mask_dir = Path("/home/chest_ct/code/data/lung_masked_cts")
output_mask_dir.mkdir(parents=True, exist_ok=True)

CANDIDATE_FILTER_CONFIG = {
    "volume_mm3": (9.0, 320.0),        # min/max nodule volume
    "equiv_diameter_mm": (2.0, 8.5),
    "sphericity": (0.55, 1.05),
    "elongation": (1.0, 4.0),
    "extent": (0.1, 1.0),
    "hu_mean": (-910.0, -220.0),
    "surface_area_mm2": (13.0, 340.0),
    "centroid_z_frac": (0.09, 1.3),
}

In [ ]:
# -----------------------------
# Helper functions
# -----------------------------
def keep_largest_components(binary_mask, n=2):
    labeled, num = label(binary_mask, return_num=True, connectivity=1)

    if num == 0:
        return np.zeros_like(binary_mask, dtype=bool)

    component_sizes = np.bincount(labeled.ravel())
    component_sizes[0] = 0

    largest_labels = np.argsort(component_sizes)[-n:]

    return np.isin(labeled, largest_labels)


def find_ct_path(filename):
    matches = list(volume_root.rglob(filename))

    if len(matches) == 0:
        return None

    return matches[0]


def lung_preprocess(ct):
    lung_candidate = (ct >= MIN_LUNG_HU) & (ct <= MAX_LUNG_HU)

    mask_clear = np.zeros_like(lung_candidate, dtype=bool)

    for k in range(lung_candidate.shape[2]):
        mask_clear[:, :, k] = clear_border(lung_candidate[:, :, k])

    lungs_only = keep_largest_components(mask_clear, n=2)

    structure = ndimage.generate_binary_structure(3, 2)

    mask_closed = ndimage.binary_closing(
        lungs_only,
        structure=structure,
        iterations=2
    )

    mask_filled = ndimage.binary_fill_holes(mask_closed)

    lungs_final = ndimage.binary_closing(
        mask_filled,
        structure=structure,
        iterations=2
    )

    lungs_final = ndimage.binary_fill_holes(lungs_final)

    ct_lung_only = ct.copy()
    ct_lung_only[~lungs_final] = -1000

    return ct_lung_only

def vessel_suppression(ct_lung_only):
    vessel_suppressed = sato(ct_lung_only, sigmas=range(1, 4), black_ridges=False)
    return vessel_suppressed

def nlog_blob_detection_3d(ct_lung_only, hu_min=MIN_NODULE_HU, hu_max=MAX_NODULE_HU):
    candidate = (ct_lung_only >= hu_min) & (ct_lung_only <= hu_max)

    min_diameter = 2
    max_diameter = 9

    filtered_blobs = []

    blob_input = ct_lung_only.copy()
    blob_input[~candidate] = 0

    blobs = blob_log(
        blob_input,
        min_sigma=2,
        max_sigma=2,
        num_sigma=1,
        threshold=0.01
    )

    blobs[:, 3] = blobs[:, 3] * np.sqrt(3)

    for z, y, x, r in blobs:
        diameter = 2 * r

        if min_diameter <= diameter <= max_diameter:
            filtered_blobs.append([z, y, x, r])

    filtered_blobs = np.array(filtered_blobs)
    return filtered_blobs, candidate

# -----------------------------
# Load filenames
# -----------------------------
with open(json_path, "r") as f:
    dataset = json.load(f)

ct_names = dataset["train"]

print("Number of CTs:", len(ct_names))

Number of CTs: 324


In [4]:
# -----------------------------
# Cell: Blob -> seed mask
# -----------------------------
def blob_to_mask(blob, shape, spacing):
    """
    Build a binary candidate mask from a LoG blob.
    blob: (z, y, x, sigma) in voxel units, sigma already in mm-equivalent radius
          (adjust indexing to match your blob_log output ordering).
    shape: CT volume shape
    spacing: voxel spacing (mm) per axis
    """
    z, y, x, sigma_mm = blob

    radius_vox = np.array(
        [sigma_mm / spacing[i] for i in range(3)]
    )

    zz, yy, xx = np.ogrid[:shape[0], :shape[1], :shape[2]]

    seed = (
        ((zz - z) / max(radius_vox[0], 1e-6)) ** 2
        + ((yy - y) / max(radius_vox[1], 1e-6)) ** 2
        + ((xx - x) / max(radius_vox[2], 1e-6)) ** 2
    ) <= 1.0

    mask = ndimage.binary_dilation(seed, iterations=2)
    return mask

# -----------------------------
# Cell: Feature extraction (refactored, reusable)
# -----------------------------
def extract_candidate_features(comp_mask, ct, spacing):
    """
    Compute the same feature set used for GT nodules, for a candidate mask.
    Returns None if the candidate is degenerate.
    """
    voxel_vol_mm3 = float(np.prod(spacing))
    voxel_count = int(comp_mask.sum())

    if voxel_count < 2:
        return None

    hu_in = ct[comp_mask]
    hu_in = hu_in[np.isfinite(hu_in)]
    if hu_in.size == 0:
        return None

    dilated = ndimage.binary_dilation(comp_mask, iterations=2)
    shell = dilated & ~comp_mask
    hu_shell = ct[shell]
    hu_shell = hu_shell[np.isfinite(hu_shell)]

    props = measure.regionprops(comp_mask.astype(np.uint8), spacing=spacing)
    if not props:
        return None
    props = props[0]

    bbox = props.bbox
    bbox_dims_mm = [
        (bbox[3] - bbox[0]) * spacing[0],
        (bbox[4] - bbox[1]) * spacing[1],
        (bbox[5] - bbox[2]) * spacing[2],
    ]

    volume_mm3 = voxel_count * voxel_vol_mm3
    equiv_diameter_mm = 2 * (3 * volume_mm3 / (4 * np.pi)) ** (1 / 3)

    try:
        verts, faces, _, _ = measure.marching_cubes(
            comp_mask.astype(np.uint8), level=0.5, spacing=spacing
        )
        surface_area_mm2 = measure.mesh_surface_area(verts, faces)
        sphere_sa = np.pi ** (1 / 3) * (6 * volume_mm3) ** (2 / 3)
        sphericity = sphere_sa / surface_area_mm2 if surface_area_mm2 > 0 else np.nan
    except Exception:
        surface_area_mm2 = np.nan
        sphericity = np.nan

    return {
        "voxel_count": voxel_count,
        "volume_mm3": volume_mm3,
        "equiv_diameter_mm": equiv_diameter_mm,
        "bbox_dim_x_mm": bbox_dims_mm[0],
        "bbox_dim_y_mm": bbox_dims_mm[1],
        "bbox_dim_z_mm": bbox_dims_mm[2],
        "elongation": max(bbox_dims_mm) / (min(bbox_dims_mm) + 1e-6),
        "extent": props.extent,
        "sphericity": sphericity,
        "surface_area_mm2": surface_area_mm2,
        "hu_mean": float(hu_in.mean()),
        "hu_std": float(hu_in.std()),
        "hu_min": float(hu_in.min()),
        "hu_max": float(hu_in.max()),
        "hu_p10": float(np.percentile(hu_in, 10)),
        "hu_p90": float(np.percentile(hu_in, 90)),
        "shell_hu_mean": float(hu_shell.mean()) if hu_shell.size else np.nan,
        "hu_contrast": (
            float(hu_in.mean() - hu_shell.mean()) if hu_shell.size else np.nan
        ),
        "centroid_z_frac": props.centroid[2] / ct.shape[2],
    }

# -----------------------------
# Cell: Rule-based filter
# -----------------------------
def passes_filter(features, config):
    if features is None:
        return False

    for key, (lo, hi) in config.items():
        val = features.get(key, np.nan)
        if not np.isfinite(val):
            return False
        if not (lo <= val <= hi):
            return False

    return True

# -----------------------------
# Cell: Build multi-instance mask + save
# -----------------------------
def build_instance_mask(accepted_masks, shape, affine, header, out_path):
    """
    accepted_masks: list of boolean arrays (one per accepted candidate)
    Saves a 4D (F,H,W,D) NIfTI matching the GT mask convention,
    channel 0 = nodule instance labels.
    """
    instance_mask = np.zeros(shape, dtype=np.int16)

    for idx, m in enumerate(accepted_masks, start=1):
        instance_mask[m & (instance_mask == 0)] = idx

    out_mask = instance_mask[np.newaxis, ...]  # (1, H, W, D) -> channel 0

    nib.save(
        nib.Nifti1Image(out_mask.astype(np.int16), affine, header),
        out_path,
    )

In [ ]:
# -----------------------------
# Cell: Updated main loop
# -----------------------------
missing_ct = []
missing_mask = []
failed = []
candidate_records = []

output_mask_dir.mkdir(parents=True, exist_ok=True)  # define this path beforehand

# for filename in tqdm(ct_names[:5]):  # Example: process only the first 5 files
filename = "train_1387_a_2.nii.gz"  # Example: process a single file
try:
    ct_path = find_ct_path(filename)
    mask_path = mask_root / filename
    print(f"Processing {filename}...")

    if ct_path is None:
        missing_ct.append(filename)
        # continue
    if not mask_path.exists():
        missing_mask.append(filename)
        # continue

    ct_img = nib.load(str(ct_path))
    ct = ct_img.get_fdata()
    spacing = ct_img.header.get_zooms()[:3]

    print("preprocessing CT...")
    ct_lung_only = lung_preprocess(ct)

    print("Vessel suppression using Sato filter...")
    ct_lung_only = vessel_suppression(ct_lung_only)

    print("detecting blobs...")
    blobs, _ = nlog_blob_detection_3d(ct_lung_only)
    print(f"Detected {len(blobs)} blobs in {filename}")

    accepted_masks = []

    for blob in tqdm(blobs):
        comp_mask = blob_to_mask(blob, ct.shape, spacing)
        features = extract_candidate_features(comp_mask, ct, spacing)

        if passes_filter(features, CANDIDATE_FILTER_CONFIG):
            accepted_masks.append(comp_mask)
            candidate_records.append({"case": filename, **features})

    print(f"Accepted masks for {filename}: {len(accepted_masks)}")

    if accepted_masks:
        build_instance_mask(
            accepted_masks,
            shape=ct.shape,
            affine=ct_img.affine,
            header=ct_img.header,
            out_path=output_mask_dir / filename,
        )

    print(f"{filename}: {len(blobs)} blobs -> {len(accepted_masks)} accepted")

except Exception as e:
    failed.append((filename, str(e)))

print("Done")
print("Missing CTs:", len(missing_ct))
print("Missing masks:", len(missing_mask))
print("Failed:", len(failed))

candidates_df = pd.DataFrame(candidate_records)
print(f"Total accepted candidates: {len(candidates_df)}")

Processing train_1387_a_2.nii.gz...
preprocessing CT...
detecting blobs...
Detected 5429 blobs in train_1387_a_2.nii.gz


  0%|          | 0/5429 [00:00<?, ?it/s]

KeyboardInterrupt: 